In [10]:
import numpy as np
import pandas as pd


In [ ]:
# Variables
TRAIN_DATA_PATH = "train.csv"
TEST_DATA_PATH = "test.csv"

In [12]:
data = pd.read_csv(TRAIN_DATA_PATH)
data = np.array(data)
print(f"Data_Shape : {data.shape} " )
# x_dev = data[:21000 , 1:]
# y_dev = data[:21000,0:1]
# x_test = data[21000:,1:]
# y_test = data[21000: , 0:1]
train_data = data[0:41000, 1:]
train_label = data[0:41000,0:1]

validataion_data = data[41000: , 1: ]
validation_label = data[41000: , 0:1]
print(train_data.shape, train_label.shape)

Data_Shape : (42000, 785) 
(41000, 784) (41000, 1)


In [13]:
rand = np.random.default_rng(seed=42)
w1 = rand.random((800, 784)) * 0.01
b1 = np.zeros(800)
w2 = rand.random((10,800)) * 0.01
b2 = np.zeros(10)
learning_rate = 0.0001

In [14]:
def softmax(z:np.array):
    z = z - np.max(z) 
    exp = np.exp(z)
    return exp/exp.sum()

def relu (z:np.array):
    return np.maximum(0,z)

def to_onehot(label, classes = 10):
    y = np.zeros(classes)
    y[label] = 1 
    return y
def grad_relu (z : np.array):
    return (z > 0).astype(float)
def entropy_loss(y:np.array , y_hat):
   return -np.sum(y * np.log(y_hat + 1e-9))
def save_model(filepath = "fnn_12_neuron_weights.npz"):
    np.savez(filepath, w1=w1, b1=b1, w2=w2, b2=b2)
    print(f"Model saved to {filepath}")
def load_model(filepath="fnn_12_neuron_weights.npz"):
    global w1, w2, b1, b2
    data = np.load(filepath)
    w1 = data['w1']
    b1 = data['b1']
    w2 = data['w2']
    b2 = data['b2']
    print(f"Model loaded from {filepath}")



In [15]:
def predict(x:np.array):
    x = x.reshape(-1, 1)
    z1 = w1 @ x + b1.reshape(-1, 1)
    a1 = relu(z1)
    z2 = w2 @ a1 + b2.reshape(-1, 1)
    y_hat = softmax(z2)
    return np.argmax(y_hat) 
def train_network( x:np.array , actual_class):
    global w1, w2, b1,b2
    x = x.reshape(-1,1) # shape : 784 , 1
    z1 = w1 @ x + b1.reshape(-1,1)  
    a1= relu(z1)
    z2= w2 @ a1 + b2.reshape(-1,1)
    y_hat = softmax(z2)
    y_actual = to_onehot(actual_class).reshape(-1,1)
    dlz2 = y_hat - y_actual 
    dlb2 = dlz2.flatten()
    dlw2 = dlz2 @ a1.T
    dla1 = w2.T @ dlz2
    dlz1 = dla1 * grad_relu(z1)
    dlw1 = dlz1 @ x.T
    dlb1 = dlz1.flatten()
    w1 = w1 - (learning_rate * dlw1)
    w2 = w2 - (learning_rate * dlw2)
    b1 = b1 - (learning_rate * dlb1)
    b2 = b2 - (learning_rate * dlb2) 
  
    loss = entropy_loss(y = y_actual , y_hat=y_hat)
    return loss

In [16]:
def main():
    epochs = 40
    train_data_len = len(train_data)
    for epoch in range(0,epochs): 
        epoch_loss = 0
        for index, data in enumerate(train_data):
            loss = train_network(x = data, actual_class= train_label[index][0])
            epoch_loss += loss
        avg_loss = epoch_loss /train_data_len 
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {avg_loss:.6f}")
    
    save_model()
main()

Epoch 1/40 - Loss: 0.348215
Epoch 2/40 - Loss: 0.145922
Epoch 3/40 - Loss: 0.108359
Epoch 4/40 - Loss: 0.086595
Epoch 5/40 - Loss: 0.073791
Epoch 6/40 - Loss: 0.063397
Epoch 7/40 - Loss: 0.055534
Epoch 8/40 - Loss: 0.052052
Epoch 9/40 - Loss: 0.053859
Epoch 10/40 - Loss: 0.038039
Epoch 11/40 - Loss: 0.042496
Epoch 12/40 - Loss: 0.035295
Epoch 13/40 - Loss: 0.037674
Epoch 14/40 - Loss: 0.033585
Epoch 15/40 - Loss: 0.037617
Epoch 16/40 - Loss: 0.034299
Epoch 17/40 - Loss: 0.024370
Epoch 18/40 - Loss: 0.029558
Epoch 19/40 - Loss: 0.041180
Epoch 20/40 - Loss: 0.031906
Epoch 21/40 - Loss: 0.027312
Epoch 22/40 - Loss: 0.030124
Epoch 23/40 - Loss: 0.025211
Epoch 24/40 - Loss: 0.024109
Epoch 25/40 - Loss: 0.026740
Epoch 26/40 - Loss: 0.025916
Epoch 27/40 - Loss: 0.028312
Epoch 28/40 - Loss: 0.025100
Epoch 29/40 - Loss: 0.018080
Epoch 30/40 - Loss: 0.027069
Epoch 31/40 - Loss: 0.025597
Epoch 32/40 - Loss: 0.026727
Epoch 33/40 - Loss: 0.030959
Epoch 34/40 - Loss: 0.034654
Epoch 35/40 - Loss: 0.0

In [17]:
def test_model(test_data, test_labels=None):
    predictions = []
    correct = 0
    total = len(test_data)
    
    print(f"\nTesting on {total} samples...")
    print("="*60)
    
    for index, data in enumerate(test_data):
        pred = predict(data)
        predictions.append(pred)
        if test_labels is not None:
            actual = test_labels[index][0]
            is_correct = pred == actual
            correct += is_correct
            
            # Print every 100th sample or incorrect predictions
            if (index + 1) % 100 == 0 or not is_correct:
                status = "✓" if is_correct else "✗"
                print(f"Sample {index+1:4d}: Predicted {pred} | Actual {actual} {status}")
    print("="*60)
    
    if test_labels is not None:
        accuracy = (correct / total) * 100
        print(f"\nAccuracy: {accuracy:.2f}% ({correct}/{total} correct)")
    else:
        print(f"\nTotal predictions: {total}")
    
    return predictions

load_model()
prediction = test_model(validataion_data , test_labels= validation_label)

Model loaded from fnn_12_neuron_weights.npz

Testing on 1000 samples...
Sample   21: Predicted 8 | Actual 2 ✗
Sample   69: Predicted 1 | Actual 5 ✗
Sample  100: Predicted 7 | Actual 7 ✓
Sample  145: Predicted 3 | Actual 7 ✗
Sample  173: Predicted 0 | Actual 5 ✗
Sample  200: Predicted 5 | Actual 5 ✓
Sample  222: Predicted 4 | Actual 9 ✗
Sample  237: Predicted 8 | Actual 0 ✗
Sample  275: Predicted 5 | Actual 8 ✗
Sample  276: Predicted 6 | Actual 1 ✗
Sample  295: Predicted 8 | Actual 5 ✗
Sample  300: Predicted 8 | Actual 8 ✓
Sample  349: Predicted 2 | Actual 6 ✗
Sample  359: Predicted 9 | Actual 4 ✗
Sample  400: Predicted 6 | Actual 6 ✓
Sample  433: Predicted 4 | Actual 7 ✗
Sample  455: Predicted 0 | Actual 5 ✗
Sample  500: Predicted 7 | Actual 7 ✓
Sample  529: Predicted 9 | Actual 5 ✗
Sample  600: Predicted 6 | Actual 6 ✓
Sample  641: Predicted 4 | Actual 9 ✗
Sample  644: Predicted 3 | Actual 5 ✗
Sample  692: Predicted 1 | Actual 4 ✗
Sample  700: Predicted 9 | Actual 9 ✓
Sample  744: Pre

In [ ]:
def save_predictions_to_csv(predictions, output_filename="predictions.csv"):

    # Create DataFrame with ImageID (0-indexed or 1-indexed)
    df = pd.DataFrame({
        'ImageID': range(1, len(predictions) + 1),  # 1-indexed
        'Label': predictions
    })
    
    # Save to CSV
    df.to_csv(output_filename, index=False)
    print(f"✓ Predictions saved to {output_filename}")
    print(f"  Total predictions: {len(predictions)}")
    print(f"\nFirst 5 rows:")
    print(df.head())

test_data = pd.read_csv(TEST_DATA_PATH)
test_data = np.array(test_data)
load_model()
prediction = test_model(test_data=test_data)
save_predictions_to_csv(predictions=prediction)


Model loaded from fnn_12_neuron_weights.npz

Testing on 28000 samples...

Total predictions: 28000
✓ Predictions saved to predictions.csv
  Total predictions: 28000

First 5 rows:
   ImageID  Label
0        1      2
1        2      0
2        3      9
3        4      9
4        5      3
